# Machine Learning - Tecnicas Avancadas
## Hierarchical Clustering, Categorical Data, Bootstrap, Cross Validation and AUC-ROC

[◀ Anterior](29_ML_Classificacao.ipynb) | [📖 Indice](00_Index.ipynb) | [Proximo ▶](31_DSA_Estruturas.ipynb)

---

## 🎯 Objetivos de Aprendizagem

- Aplicar clustering hierarquico com dendrogramas
- Processar dados categoricos (variaveis qualitativas)
- Implementar Bootstrap Aggregation (Bagging)
- Validar modelos com Cross Validation (validacao cruzada)
- Avaliar classificadores com a curva AUC-ROC

## 📝 Introducao

Neste modulo avancado, exploramos tecnicas que elevam a qualidade e confiabilidade dos modelos de Machine Learning:

- **Clustering Hierarquico**: cria uma hierarquia de clusters, visualizada com dendrogramas
- **Dados Categoricos**: como transformar variaveis como "cor", "cidade" ou "categoria" em numeros
- **Bootstrap Aggregation (Bagging)**: tecnica de ensemble que reduz variancia treinando modelos em subconjuntos
- **Cross Validation**: dividir dados em K folds para avaliacao mais robusta
- **AUC-ROC**: metrica avancada para classificadores que considera todos os limiares

## 📚 Explicacao Detalhada

### Clustering Hierarquico

Diferente do K-Means, nao exige especificar o numero de clusters. Dois tipos:
- **Aglomerativo (bottom-up)**: cada ponto comeca como cluster, vai unindo
- **Divisivo (top-down)**: todos pontos em um cluster, vai dividindo

O resultado e visualizado com um **dendrograma**.

```python
from scipy.cluster.hierarchy import dendrogram, linkage
```

### Dados Categoricos

Algoritmos de ML trabalham com numeros. Precisamos converter categorias:
- **Label Encoding**: cada categoria vira um numero (0, 1, 2...)
- **One-Hot Encoding**: cada categoria vira uma coluna binaria

### Bootstrap Aggregation (Bagging)

Treina o mesmo algoritmo em varios subconjuntos bootstrap (amostras com reposicao) e combina os resultados (media para regressao, votacao para classificacao).

### Cross Validation (K-Fold)

Divide os dados em K partes iguais. Treina em K-1 e testa na restante, repetindo para cada fold.

### AUC-ROC Curve

- **ROC**: grafico de True Positive Rate vs False Positive Rate em varios limiares
- **AUC**: area sob a curva ROC — 1.0 e perfeito, 0.5 e aleatorio

```python
from sklearn.metrics import roc_auc_score, roc_curve
```

## 💻 Exemplos

In [ ]:
# Exemplo 1: Clustering Hierarquico com Dendrograma
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.datasets import make_blobs

np.random.seed(42)
X, _ = make_blobs(n_samples=50, centers=4, random_state=42)

Z = linkage(X, method='ward')

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], s=50, alpha=0.7)
plt.title('Dados Originais')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
dendrogram(Z, truncate_mode='level', p=4)
plt.title('Dendrograma (Clustering Hierarquico)')
plt.xlabel('Amostras')
plt.ylabel('Distancia')
plt.axhline(y=15, color='r', linestyle='--', label='Corte sugerido')
plt.legend()

plt.tight_layout()
plt.show()

clusters = fcluster(Z, t=15, criterion='distance')
print(f"Numero de clusters encontrados: {len(set(clusters))}")

In [ ]:
# Exemplo 2: Comparacao K-Means vs Hierarquico
from sklearn.cluster import KMeans, AgglomerativeClustering

np.random.seed(42)
X_moons, _ = make_blobs(n_samples=300, centers=3, cluster_std=1.5, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], s=20, alpha=0.7)
axes[0].set_title('Dados Originais')

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X_moons)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_km, cmap='viridis', s=20)
axes[1].set_title('K-Means (K=3)')

agg = AgglomerativeClustering(n_clusters=3)
labels_agg = agg.fit_predict(X_moons)
axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_agg, cmap='viridis', s=20)
axes[2].set_title('Hierarquico (n=3)')

plt.tight_layout()
plt.show()

In [ ]:
# Exemplo 3: Processamento de Dados Categoricos (One-Hot Encoding)
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

dados = pd.DataFrame({
    'Cidade': ['SP', 'RJ', 'BH', 'SP', 'RJ', 'POA', 'BH', 'SP'],
    'Cor': ['Azul', 'Vermelho', 'Verde', 'Azul', 'Verde', 'Azul', 'Vermelho', 'Verde'],
    'Tamanho': ['P', 'M', 'G', 'M', 'P', 'G', 'M', 'P']
})

print("Dados originais:")
print(dados)

# One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(dados)
colunas = encoder.get_feature_names_out(dados.columns)

df_encoded = pd.DataFrame(encoded, columns=colunas)
print(f"\nApos One-Hot Encoding ({len(colunas)} colunas):")
print(df_encoded)

# Label Encoding
le = LabelEncoder()
df_label = dados.copy()
for col in dados.columns:
    df_label[col] = le.fit_transform(dados[col])

print(f"\nApos Label Encoding (MESMO numero de colunas):")
print(df_label)

In [ ]:
# Exemplo 4: Bootstrap Aggregation (Bagging)
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

dt_solo = DecisionTreeClassifier(random_state=42)
dt_solo.fit(X_train, y_train)

bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    random_state=42
)
bagging.fit(X_train, y_train)

print(f"Decision Tree unica:       {dt_solo.score(X_test, y_test):.4f}")
print(f"Bagging (100 arvores):      {bagging.score(X_test, y_test):.4f}")

In [ ]:
# Exemplo 5: Cross Validation (K-Fold)
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

modelo = LogisticRegression(random_state=42, max_iter=200)

scores = cross_val_score(modelo, X_scaled, y, cv=5)
print(f"Scores por fold: {scores}")
print(f"Media: {scores.mean():.4f}")
print(f"Desvio Padrao: {scores.std():.4f}")

scores_k10 = cross_val_score(modelo, X_scaled, y, cv=10)
print(f"\nCom K=10 folds:")
print(f"Media: {scores_k10.mean():.4f}")
print(f"Desvio Padrao: {scores_k10.std():.4f}")

In [ ]:
# Exemplo 6: Curva AUC-ROC
from sklearn.metrics import roc_curve, roc_auc_score, auc

y_scores = modelo.fit(X_train, y_train).predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc_score = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Aleatorio (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.1, color='blue')
plt.xlabel('False Positive Rate (1 - Especificidade)')
plt.ylabel('True Positive Rate (Sensibilidade)')
plt.title('Curva ROC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.show()

print(f"AUC-ROC: {roc_auc_score(y_test, y_scores):.4f}")

In [ ]:
# Exemplo 7: Comparacao de AUC-ROC entre modelos
from sklearn.ensemble import RandomForestClassifier

modelos = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=200),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
}

plt.figure(figsize=(8, 6))

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    if hasattr(modelo, 'predict_proba'):
        y_scores = modelo.predict_proba(X_test)[:, 1]
    else:
        y_scores = modelo.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f'{nome} (AUC={auc_score:.4f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Comparacao de Curvas ROC')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## ✅ Exercicios Resolvidos

In [ ]:
# Exercicio 1: Clustering hierarquico e silhueta
from sklearn.metrics import silhouette_score

scores = []
for k in range(2, 8):
    agg = AgglomerativeClustering(n_clusters=k)
    labels = agg.fit_predict(X)
    scores.append(silhouette_score(X, labels))

plt.plot(range(2, 8), scores, 'bo-')
plt.xlabel('Numero de clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score para Clustering Hierarquico')
plt.grid(True, alpha=0.3)
plt.show()
print(f"Melhor silhouette: {max(scores):.4f} com K={np.argmax(scores)+2}")

In [ ]:
# Exercicio 2: Cross Validation comparando modelos
modelos_cv = {
    'LogReg': LogisticRegression(max_iter=200, random_state=42),
    'DecisionTree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=50, random_state=42)
}

for nome, m in modelos_cv.items():
    scores = cross_val_score(m, X_scaled, y, cv=5)
    print(f"{nome:<20s}: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

## 📋 Exercicios Propostos

### Facil

1. Crie um dendrograma para um conjunto de dados de 30 pontos com 3 clusters naturais.
2. Aplique One-Hot Encoding a um dataframe com colunas categoricas de cores e tamanhos.
3. Execute Cross Validation com 5 folds em um modelo de regressao logistica.
4. Plote a curva ROC de um classificador e calcule o AUC.

### Medio

5. Compare K-Means com Clustering Hierarquico em um dataset nao esferico (use `make_moons`).
6. Crie um pipeline com `OneHotEncoder` + `StandardScaler` + `LogisticRegression` usando `ColumnTransformer`.
7. Implemente Bagging com 10, 50, 100 e 200 estimadores e compare as acuracias.
8. Use `cross_val_score` para comparar 3 algoritmos diferentes com 10-fold CV.

### Dificil

9. Implemente um ensemble manual (voting classifier) que combina Logistic Regression, Decision Tree e KNN usando votacao majoritaria.
10. Crie uma funcao que plota learning curves (curvas de aprendizado) mostrando score de treino e validacao conforme o tamanho do dataset aumenta.

## 🏆 Desafios

1. **Pipeline Completo de ML**: Crie um pipeline end-to-end para o dataset Titanic: (1) pre-processamento de dados categoricos e missing values, (2) feature engineering, (3) treinamento com GridSearchCV, (4) avaliacao com confusion matrix, classification report e curva ROC. Documente cada etapa.

2. **Ensemble Avancado**: Implemente e compare 3 metodos de ensemble: Bagging (Random Forest), Boosting (Gradient Boosting) e Stacking (combinando Logistic Regression, KNN, Decision Tree como base e Logistic Regression como meta-modelo). Use cross-validation e AUC-ROC como metrica.

## 📌 Resumo

- **Clustering Hierarquico**: dendrograma, nao precisa especificar K
- **One-Hot Encoding**: transforma categorias em colunas binarias
- **Label Encoding**: transforma categorias em numeros (cuidado com ordinalidade!)
- **Bagging**: ensemble que treina modelos em subconjuntos bootstrap
- **Cross Validation**: divide dados em K folds para avaliacao robusta
- **AUC-ROC**: area sob a curva ROC — 0.5 = aleatorio, 1.0 = perfeito
- **Silhouette Score**: metrica para qualidade de clusters (-1 a 1)

## 💡 Curiosidades

- **Bagging** (Bootstrap AGGregatING) foi proposto por **Leo Breiman** em 1996. E a base do Random Forest.
- O termo **bootstrap** vem da expressao "pull oneself up by one's bootstraps" (erguer-se pelos proprios cadarcos) — a ideia de criar algo do nada.
- **ROC** significa "Receiver Operating Characteristic", termo que vem da Segunda Guerra Mundial, quando era usado para analisar sinais de radar.
- **One-Hot Encoding** recebe esse nome porque apenas um bit fica "quente" (1) enquanto os outros ficam "frios" (0).
- O **silhouette score** foi proposto por Peter Rousseeuw em 1987 e e ate hoje a metrica mais popular para clustering.

## ✅ Boas Praticas

- Use `OneHotEncoder` com `drop='first'` para evitar multicolinearidade (dummy variable trap)
- Sempre use Cross Validation para avaliacao — nunca confie em um unico split treino/teste
- Para dados desbalanceados, use `StratifiedKFold` que preserva a proporcao das classes em cada fold
- AUC-ROC e melhor que acuracia para problemas com classes desbalanceadas
- Bagging funciona melhor com modelos de alta variancia (ex: Decision Trees profundas)

## ⚠️ Erros Comuns

| Erro | Exemplo | Correcao |
|------|---------|----------|
| One-Hot em muitas categorias | Explosao de dimensionalidade | Agrupar categorias raras ou usar Target Encoding |
| Label Encoding para nao-ordinal | "SP"=0, "RJ"=1, "BH"=2 sugere ordem falsa | Usar One-Hot Encoding |
| Cross Validation com series temporais | K-Fold aleatorio em dados temporais | Usar TimeSeriesSplit |
| Interpretar AUC=0.5 com modelo bom | Modelo "sabe" mas limiar esta errado | AUC e independente de limiar |
| Vazamento no pre-processamento | Fit do OneHotEncoder em todos os dados antes do split | Fit apenas no treino, transform no teste |

## 📖 Referencias

- [W3Schools - Hierarchical Clustering](https://www.w3schools.com/python/python_ml_hierarchial_clustering.asp)
- [W3Schools - Categorical Data](https://www.w3schools.com/python/python_ml_preprocessing.asp)
- [W3Schools - Bootstrap Aggregation](https://www.w3schools.com/python/python_ml_bagging.asp)
- [W3Schools - Cross Validation](https://www.w3schools.com/python/python_ml_cross_validation.asp)
- [W3Schools - AUC-ROC Curve](https://www.w3schools.com/python/python_ml_auc_roc.asp)
- [Scikit-learn: Ensemble Methods](https://scikit-learn.org/stable/modules/ensemble.html)
- [Scikit-learn: Model Selection](https://scikit-learn.org/stable/modules/model_selection.html)

---

## 📄 Creditos

**Compilado e desenvolvido por Luciano Vilas Boas Espiridiao**

📧 luciano.espiridiao@ifmg.edu.br

Baseado no site da W3Schools.

---

[◀ Anterior](29_ML_Classificacao.ipynb) | [📖 Indice](00_Index.ipynb) | [Proximo ▶](31_DSA_Estruturas.ipynb)